# Segger graph construction (v0.2.0)

This notebook explains how Segger builds its graphs, how big they can get,
why tiling is required for efficiency, and how to choose the key graph
parameters.


## Code map (v0.2.0)

- Graph construction: `segger/data/utils/heterodata.py`
- Neighbor and edge builders: `segger/data/utils/neighbors.py`
- Tiling + datasets: `segger/data/tiling.py`, `segger/data/tile_dataset.py`,
  `segger/data/partition/*`
- Orchestration: `segger/data/data_module.py` (ISTDataModule)
- Math overview: `docs/MATH.md`


## Graphs at a glance

Segger builds a heterogeneous PyG graph with:

- Node types:
  - `tx` = transcripts
  - `bd` = boundaries (cell or nucleus polygons)
- Edge types:
  - (`tx`, `neighbors`, `tx`) : local transcript graph (kNN within max distance)
  - (`tx`, `belongs`, `bd`)   : ground-truth links for training
  - (`tx`, `neighbors`, `bd`) : candidate links for prediction
- Optional edge type (alignment loss):
  - (`tx`, `attracts`, `tx`) : subset of tx-tx edges labeled by ME gene pairs


## How large are these graphs? (10x Xenium 5k example)

The repo includes a Xenium Human Colon Add-on dataset (5k panel). The file
`../../data/human_CRC_real/Xenium_V1_Human_Colon_Cancer_P1_CRC_Add_on_FFPE_outs/metrics_summary.csv`
contains summary counts. The code below reads that summary and prints the
transcript and cell counts so we can estimate node and edge sizes.

Vendor context (10x + CosMx):
- **10x Xenium Prime 5K** panels target 5,000 genes with up to 100 add-on genes.
  See the 10x panel overview here: [Xenium 5,000-gene panels](https://www.10xgenomics.com/products/xenium-5k-panel).
- A public **Xenium Prime 5K** dataset (FFPE human cervical cancer) reports
  **89,640,525** high-quality decoded transcripts and **840,387** cells detected — a
  useful scale reference: [10x dataset summary metrics](https://www.10xgenomics.com/datasets/xenium-prime-ffpe-human-cervical-cancer).
- **NanoString CosMx 6K** pushes plex higher: the Human Frontal Cortex dataset uses
  a **6,078-plex** panel ("over 6,000 genes"): [CosMx 6K dataset](https://nanostring.com/products/cosmx-spatial-molecular-imager/ffpe-dataset/human-frontal-cortex-ffpe-dataset/).
- **CosMx Whole Transcriptome** datasets capture **>18,000 genes** in a single run:
  [CosMx whole transcriptome dataset](https://nanostring.com/products/cosmx-spatial-molecular-imager/ffpe-dataset/cosmx-human-whole-transcriptome-colon-dataset/).


In [ ]:
from pathlib import Path
import csv

metrics_path = Path(
    "../../data/human_CRC_real/Xenium_V1_Human_Colon_Cancer_P1_CRC_Add_on_FFPE_outs/metrics_summary.csv"
)

if metrics_path.exists():
    row = next(csv.DictReader(metrics_path.open()))
    n_tx = int(float(row["total_high_quality_decoded_transcripts"]))
    n_cells = int(float(row["num_cells_detected"]))
    frac_assigned = float(row.get("fraction_transcripts_assigned", "nan"))
    panel = row.get("panel_name", "unknown")
    region = row.get("region_name", "unknown")

    print(f"panel: {panel}")
    print(f"region: {region}")
    print(f"transcripts (high quality): {n_tx:,}")
    print(f"cells detected: {n_cells:,}")
    if frac_assigned == frac_assigned:
        print(f"fraction transcripts assigned: {frac_assigned:.3f}")
else:
    print("metrics_summary.csv not found; update metrics_path to your dataset.")


In [ ]:
import math

# Defaults from ISTDataModule (v0.2.0)
k_tx = 5
max_dist_tx = 5.0
k_pred = 3
scale_factor = 2.0
max_nodes_per_tile = 50_000

if "n_tx" in globals():
    # Boundary nodes are roughly the number of cells (or nuclei) in this dataset.
    n_nodes = n_tx + n_cells

    # Edge count estimates (directed upper bounds)
    edges_tx_tx = n_tx * k_tx
    edges_belongs = int(n_tx * frac_assigned) if frac_assigned == frac_assigned else n_tx
    edges_pred = edges_belongs  # rough guess for cell/nucleus mode

    total_edges = edges_tx_tx + edges_belongs + edges_pred

    # Edge index storage (int64) is 2 * 8 bytes per edge
    edge_index_gb = edges_tx_tx * 16 / (1024 ** 3)

    tiles_est = math.ceil(n_nodes / max_nodes_per_tile)

    print(f"nodes (tx + bd): {n_nodes:,}")
    print(f"tx-tx edges (<= n_tx * k): {edges_tx_tx:,}")
    print(f"tx-bd belongs edges (~assigned tx): {edges_belongs:,}")
    print(f"tx-bd prediction edges (rough): {edges_pred:,}")
    print(f"total edges (rough): {total_edges:,}")
    print(f"tx-tx edge_index size (rough, int64): {edge_index_gb:.2f} GB")
    print(f"tiles if {max_nodes_per_tile:,} nodes/tile: ~{tiles_est:,}")


## How does this compare to other graphs?

Classic GNN benchmark graphs are tiny by comparison:
- **Planetoid** citation datasets (Cora/CiteSeer/PubMed) are only **2.7k–19.7k nodes**
  and **9k–88k edges**. [PyG Planetoid stats](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.datasets.Planetoid.html)

Large-scale AI benchmarks can reach the billion‑edge regime:
- **OGB ogbn-papers100M** has **111,059,956 nodes** and **1,615,685,872 edges**.
  [OGB node property datasets](https://ogb.stanford.edu/docs/nodeprop/)

A Xenium 5k slide with tens of millions of transcripts can push the **10^8+ edge**
range, which puts Segger graphs in the same computational regime as the largest
benchmark graphs. This is why Segger emphasizes efficiency and **tiling**.


## Graph construction pipeline (v0.2.0)

1. **Load and standardize inputs**
   - Transcripts and boundaries are loaded and normalized (platform auto-detect
     or SpatialData). Quality filtering happens here (e.g., Xenium QV).

2. **Build embeddings and clusters (AnnData)**
   - `setup_anndata` computes cell and gene embeddings, plus cluster labels.

3. **Assemble HeteroData** (`setup_heterodata`)
   - `tx` node features:
     - `x` = gene encoding (index), `cluster`, `index`
     - `pos` and `geometry` = 2D coordinates (x, y)
     - `geometry_3d` = (x, y, z) only if `use_3d` is enabled
   - `bd` node features:
     - `x` = cell embedding, `cluster`, `index`
     - `pos` and `geometry` = 2D boundary centroid positions

4. **Build edges** (see `segger/data/utils/neighbors.py`)
   - `tx -> neighbors -> tx`
     - KDTree kNN with `transcripts_graph_max_k` and `transcripts_graph_max_dist`
     - Optional cosine similarity on edges if `compute_tx_similarities=True`
   - `tx -> belongs -> bd`
     - Ground-truth edges for training (segmentation mask)
   - `tx -> neighbors -> bd`
     - Prediction graph:
       - `cell` or `nucleus` mode: polygon containment after scaling by
         `prediction_graph_scale_factor`
       - `uniform` mode: kNN to boundary centroids
   - Optional `tx -> attracts -> tx` edges for alignment loss


## Tiling and batching (in-memory)

Segger v0.2.0 builds the full graph **once in memory** and then tiles it for
training and prediction:

- **Tiling** is built over combined `tx.pos` + `bd.pos` positions.
  - `QuadTreeTiling` (adaptive, default) caps nodes per tile.
  - `SquareTiling` (benchmarking only) uses fixed side length.
- **TileFitDataset** partitions the graph and adds a `mask`:
  - The mask is True for nodes inside an **inner margin**.
  - Losses are computed only on masked nodes, reducing boundary artifacts.
- **PartitionSampler** packs tiles into batches by **edge count** using
  `edges_per_batch`.
- **TilePredictDataset** adds an **outer margin** to include neighbors across
  tile boundaries and stores `predict_mask` so predictions can be stitched
  without double-counting.

This is the core reason large Xenium graphs are feasible: efficiency and tiling.


## Graph-related parameters (cheat sheet)

Defaults below are from `ISTDataModule` in v0.2.0. Units are the same as your
input coordinates (typically microns for Xenium).

| Parameter | Default | What it controls | Notes |
| --- | --- | --- | --- |
| `transcripts_graph_max_k` | 5 | Max kNN neighbors per transcript | Caps degree |
| `transcripts_graph_max_dist` | 5.0 | Radius limit for tx-tx edges | Acts like a distance cutoff |
| `segmentation_graph_mode` | `nucleus` | Which compartments define positive edges | `cell` includes cytoplasm |
| `prediction_graph_mode` | `cell` | How tx-bd candidates are built | `uniform` uses kNN to centroids |
| `prediction_graph_max_k` | 3 | k in uniform prediction mode | Ignored in cell/nucleus mode |
| `prediction_graph_scale_factor` | 2.0 | Polygon scale for cell/nucleus mode | >1 expands, <1 shrinks |
| `tiling_mode` | `adaptive` | Quadtree vs square tiling | Square is for benchmarking |
| `tiling_nodes_per_tile` | 50_000 | Max nodes per adaptive tile | Includes tx + bd |
| `tiling_side_length` | 250.0 | Side length for square tiles | Benchmarking only |
| `tiling_margin_training` | 20.0 | Inner margin for training masks | Must cover neighbor radius |
| `tiling_margin_prediction` | 20.0 | Outer margin for prediction tiles | Prevents boundary loss |
| `edges_per_batch` | 1_000_000 | Max edges per batch | Used by PartitionSampler |
| `use_3d` | `auto` | 2D vs 3D neighbor distances | Only affects graph building |
| `min_qv` | None | Transcript quality filtering | Xenium default if None |

Notes:
- `prediction_graph_max_dist` exists in `setup_prediction_graph` but is not
  currently wired in `ISTDataModule`.
- `segmentation_graph_negative_edge_rate` is documented but the current
  negative sampling happens inside the model (1 negative per positive).


## 3D behavior

- `use_3d` can be `"auto"`, `True`, or `False`.
- When enabled and a valid z column exists:
  - tx-tx kNN uses (x, y, z) distances
  - `tx.geometry_3d` is stored
  - boundary nodes remain 2D (polygons are 2D)
- Prediction graph:
  - `uniform` mode can use 3D distances (bd centroids get z=0)
  - `cell` / `nucleus` modes are always 2D polygon containment
- Tiling is always 2D (based on `geometry` / `pos`).

Why 3D matters (motivation):
- The **ovrlpy** project and the accompanying preprint *"2D, or not 2D? Investigating
  Vertical Signal Integrity of Tissue Slices"* show that 2D tissue slices are
  projections of 3D structure and can contain **vertical overlaps** that lead to
  signal inconsistencies and spatial doublets. ovrlpy uses **x, y, z** coordinates
  and 3D visualizations to detect these overlaps. [ovrlpy docs](https://ovrlpy.readthedocs.io/latest/)
  and [bioRxiv preprint](https://doi.org/10.1101/2025.01.13.632601).


## What changed since segger-0.1.0 (graph building)

**v0.1.0**
- Graphs were built **per tile on disk** (create_dataset / STTile pipeline).
- Parameters used older names like `k_tx`, `k_bd`, `dist_tx`, and
  `tile_size` / `tile_width` / `tile_height`.
- Training consumed pre-written `.pt` tile files and did not keep a full
  graph in memory.

**v0.2.0**
- Graphs are built **once in memory** as a single HeteroData object.
- Tiling is applied on the in-memory graph (quadtree by default).
- Batching is done by **edge count** via `PartitionSampler`.
- Prediction can keep the graph on GPU and uses tile overlap stitching.
- Parameter names are standardized (e.g., `transcripts_graph_max_k`,
  `prediction_graph_scale_factor`).
- Prediction graph modes (`cell`, `nucleus`, `uniform`) are explicit and
  polygon scaling is built in.
- 3D-aware neighbor building is supported (`use_3d`).


## Choosing parameters (rules of thumb)

### k and radius (tx-tx graph)
- **Dense tissue (lots of transcripts per area):** keep `transcripts_graph_max_k`
  small (3 to 6) and a smaller `transcripts_graph_max_dist`.
- **Sparse tissue:** increase the distance cutoff so nodes are not isolated.
- **Sanity check:** measure the fraction of tx nodes with degree 0. If it is
  high, increase `max_dist` or `max_k`.

### Prediction graph mode and scale factor
- **Use `cell` mode** when you have reliable cell boundaries.
- **Use `nucleus` mode** when you want conservative assignments or only nuclei
  are trustworthy.
- **Use `uniform` mode** for 3D data or when boundaries are missing.
- Increase `prediction_graph_scale_factor` if many transcripts fall just
  outside boundaries. Decrease it if you see cross-cell confusion.

### Tiling and memory
- `tiling_nodes_per_tile` trades memory for context. If you OOM, lower it.
  If training is too slow and memory allows, raise it.
- `edges_per_batch` should fit GPU memory. Reduce on OOM; increase if GPU
  utilization is low.
- **Margins:** set `tiling_margin_training` and `tiling_margin_prediction` to
  at least the tx-tx distance cutoff (and consider the polygon scale factor).

### 3D
- Turn `use_3d` on only when z is meaningful and consistent.
- If z is slice index or noisy, keep 2D and use smaller margins.


## Other details and gotchas

- `prediction_graph_max_k` only affects `uniform` mode. In `cell` or `nucleus`
  mode, edges come from polygon containment and are not k-limited.
- `transcripts_graph_max_dist` is a hard distance cutoff; if it is too small,
  the tx graph can become disconnected.
- The training loss uses the **mask** created by `TileFitDataset`, so margin
  settings directly affect which nodes contribute to loss.
- If you enable alignment loss, extra tx-tx edges are computed and stored
  as (`tx`, `attracts`, `tx`).
- The `use_3d` flag affects neighbor search and features, not the model
  architecture itself.
